**Step 1: Install Mage in a Python 3.12 environment**

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 /content/mage_env
!uv pip install -q --python /content/mage_env mage-ai scikit-learn

import os
os.environ['PATH'] = '/content/mage_env/bin:' + os.environ['PATH']
!mage --help

                                                                                
 Usage: mage [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ init                           Initialize Mage project.                      │
│ start                          Start Mage server and UI.                     │
│ run                       

**Step 2:Get your project**

In [ ]:
import os
os.chdir('/content')
!rm -rf /content/freshcart-pipelines
!git clone https://github.com/Manijitya30/freshcart-pipeplines.git /content/freshcart-pipelines
%cd /content/freshcart-pipelines
!ls
!ls iris_project

Cloning into '/content/freshcart-pipelines'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 45 (delta 8), reused 44 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 11.08 KiB | 11.08 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/freshcart-pipelines
conftest.py  iris_project  README.md  requirements.txt	stream	tests
charts		data_loaders  __init__.py     metadata.yaml	scratchpads
custom		dbt	      interactions    pipelines		transformers
data_exporters	extensions    io_config.yaml  requirements.txt	utils


**C1: run the pipeline**

In [ ]:
%cd /content/freshcart-pipeplines
!mage run iris_project iris_classifier --test

/content/freshcart-pipeplines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
2/2 tests passed.
--------------------------------------------------------------
1/1 tests passed.
max_depth=3  test_accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows = true, columns = predicted):
            setosa  versicolor  virginica
setosa          10           0          0
versicolor       0           9          1
virginica        0           0         10

Metrics per class:
            precision  recall     f1  support
setosa          1.000     1.0  1.000       10
versicolor      1.000     0.9  0.947       10
virginica       0.909     1.0  0.952       10

Accuracy: 0.9667
saved 30 rows to iris_predicti

**C2**

In [ ]:
import pandas as pd
pd.read_csv('iris_predictions.csv').head(10)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,predicted
0,5.0,3.4,1.5,0.2,0,0
1,5.4,3.7,1.5,0.2,0,0
2,5.8,4.0,1.2,0.2,0,0
3,5.7,3.8,1.7,0.3,0,0
4,5.4,3.4,1.7,0.2,0,0
5,4.6,3.6,1.0,0.2,0,0
6,5.2,3.4,1.4,0.2,0,0
7,4.4,3.0,1.3,0.2,0,0
8,4.4,3.2,1.3,0.2,0,0
9,5.0,3.3,1.4,0.2,0,0


**C2: the tests-as-a-gate loop**

In [ ]:
%%bash
cd /content/freshcart-pipelines
for d in 1 2 3 4 5; do
  rm -f iris_predictions.csv
  echo "=== max_depth $d"
  mage run iris_project iris_classifier --test --runtime-vars "{\"max_depth\": $d}" 2>&1 | grep -E "test_accuracy|FAIL|completed"
  ls iris_predictions.csv 2>/dev/null && echo "csv CREATED" || echo "csv NOT created"
done

=== max_depth 1
max_depth=1  test_accuracy=0.6667
FAIL: test_accuracy (block: train_model)
  File "<string>", line 34, in test_accuracy
csv NOT created
=== max_depth 2
max_depth=2  test_accuracy=0.9333
Pipeline run completed.
iris_predictions.csv
csv CREATED
=== max_depth 3
max_depth=3  test_accuracy=0.9667
Pipeline run completed.
iris_predictions.csv
csv CREATED
=== max_depth 4
max_depth=4  test_accuracy=0.9333
Pipeline run completed.
iris_predictions.csv
csv CREATED
=== max_depth 5
max_depth=5  test_accuracy=0.9333
Pipeline run completed.
iris_predictions.csv
csv CREATED


**C3 (the evaluation branch)**

In [ ]:
!cat /content/freshcart-pipelines/iris_project/pipelines/iris_classifier/metadata.yaml

blocks:
- all_upstream_blocks_executed: true
  downstream_blocks:
  - validate_iris
  name: load_iris
  status: not_executed
  type: data_loader
  upstream_blocks: []
  uuid: load_iris
- all_upstream_blocks_executed: true
  downstream_blocks:
  - prepare_data
  name: validate_iris
  status: not_executed
  type: transformer
  upstream_blocks:
  - load_iris
  uuid: validate_iris
- all_upstream_blocks_executed: true
  downstream_blocks:
  - train_model
  name: prepare_data
  status: not_executed
  type: transformer
  upstream_blocks:
  - validate_iris
  uuid: prepare_data
- all_upstream_blocks_executed: true
  downstream_blocks:
  - save_results
  - evaluate_model
  name: train_model
  status: not_executed
  type: transformer
  upstream_blocks:
  - prepare_data
  uuid: train_model
- all_upstream_blocks_executed: true
  downstream_blocks: []
  name: save_results
  status: not_executed
  type: data_exporter
  upstream_blocks:
  - train_model
  uuid: save_results
- all_upstream_blocks_execut

In [ ]:
!cat /content/freshcart-pipelines/iris_metrics.csv

,precision,recall,f1,support
setosa,1.0,1.0,1.0,10
versicolor,0.9,0.9,0.9,10
virginica,0.9,0.9,0.9,10


In [ ]:
%cd /content/freshcart-pipelines
!mage run iris_project iris_classifier --test 2>&1 | tail -5
!cat iris_metrics.csv

/content/freshcart-pipelines
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.
,precision,recall,f1,support
setosa,1.0,1.0,1.0,10
versicolor,1.0,0.9,0.947,10
virginica,0.909,1.0,0.952,10


**the Mage UI graph**

In [ ]:
import subprocess, time, urllib.request
mage_server = subprocess.Popen(['/content/mage_env/bin/mage', 'start', 'iris_project'],
                               cwd='/content/freshcart-pipelines',
                               stdout=open('/content/mage.log', 'w'), stderr=subprocess.STDOUT)
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:6789'); print('Mage is running'); break
    except Exception:
        time.sleep(2)
else:
    print('Mage did not start - check /content/mage.log')

Mage is running


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess, time, re
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:6789'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
link = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
print('Open Mage here:', link[0] if link else 'Not ready yet - run this cell again')

Open Mage here: https://highest-incentive-educators-references.trycloudflare.com


**C4 evidence: the code of your own block**

In [ ]:
!cat /content/freshcart-pipelines/iris_project/transformers/validate_iris.py

"""C4 (own block): data-quality gate between load_iris and prepare_data."""
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test


@transformer
def transform(df, *args, **kwargs):
    # Pass the data through unchanged. The @test below is the actual gate.
    return df


@test
def test_no_missing_values(df, *args) -> None:
    n = int(df.isnull().sum().sum())
    assert n == 0, f'found {n} missing values'


@test
def test_three_classes(df, *args) -> None:
    classes = sorted(df['target'].unique())
    assert len(classes) == 3, f'expected exactly 3 classes, found {classes}'


In [ ]:
   mage_server.terminate()
   tunnel.terminate()